# Homework 4 - Lab Assignment
**Mateo Larrea - Music 320 - Fall 2025**

**Topic:** Spectrogram Implementation and Peak Detection

In [ ]:
# importing needed libraries! :)
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import find_peaks
%matplotlib inline

---
## Problem 1: Implement a Spectrogram Function

### Mathematical Background:

A spectrogram is a visual representation of the spectrum of frequencies in a signal as it varies with time. We compute it by:

1. **Windowing:** Breaking the signal into overlapping blocks and applying a window function
2. **DFT:** Computing the Discrete Fourier Transform of each block
3. **PSD:** Computing the Power Spectral Density from the DFT magnitude

**Key parameters:**
- Block size: $N$ samples
- Hop size: $N/2$ samples (50% overlap)
- Window: Hanning window $w_h[n]$
- Positive frequency lines: $k \frac{f_s}{N}$ for $k \in \{0,1,...,N/2-1\}$

**Power Spectral Density formula:**
$$S_{xx}[k] = |YW[k]|^2 \cdot \frac{8}{3} \cdot \frac{4}{N^2}$$

where the $\frac{8}{3}$ factor corrects for the Hanning window energy loss and $\frac{4}{N^2}$ normalizes the DFT.

### Part a: Create a Spectrogram() Function

Implement a `spectrogram(x, fs, N)` function that:
- Takes a mono input signal `x` (normalized between -1 and 1), sampling rate `fs` in Hz, and block size `N`
- Zero-pads the signal with N/2 zeros at the beginning and end
- Processes overlapping blocks with hop size N/2
- Applies Hanning window to each block
- Computes DFT and PSD for each block
- Returns tuple: `(f, t, Sxx)` where f is frequency vector, t is time vector, and Sxx is the spectrogram matrix

In [ ]:
def spectrogram(x, fs, N):

    hop_size = N // 2

    # zero-pad the signal
    x_len = len(x)
    total_padded_len = x_len + N
    remainder = (total_padded_len - N) % hop_size
    extra_zeros = hop_size - remainder if remainder != 0 else 0
    x_padded = np.concatenate([np.zeros(hop_size), x, np.zeros(hop_size + extra_zeros)])

    # calculate dimensions
    num_blocks = (len(x_padded) - N) // hop_size + 1
    f = np.arange(N // 2) * fs / N
    t = np.arange(num_blocks) * hop_size / fs
    Sxx = np.zeros((N // 2, num_blocks))

    # window creation once for efficiency
    hanning_window = np.hanning(N)

    # process each block: window, FFT, and compute PSD
    for block_idx in range(num_blocks):
        start_idx = block_idx * hop_size
        y = x_padded[start_idx : start_idx + N]
        yw = y * hanning_window
        YW = np.fft.fft(yw)
        Sxx[:, block_idx] = np.abs(YW[:N // 2])**2 * (8 / 3) * (4 / N**2)

    return f, t, Sxx

### Part b: Test Your Spectrogram() Function

Test the spectrogram function with:
1. A full-scale 1 kHz sinusoid at 44.1 kHz sampling rate
2. Plot a single column of Sxx in dBFS to verify peak location and amplitude
3. Create a colormesh visualization of the full spectrogram
4. Test with real audio: oboe recording from SQAM disk

In [ ]:
# test - 1 kHz sinusoid
fs = 44100
duration = 2.0
f0 = 1000
A = 1.0
N = 4096

t_signal = np.arange(0, duration, 1/fs)
x_1khz = A * np.sin(2 * np.pi * f0 * t_signal)

# compute spectrogram and convert to dBFS
f, t, Sxx = spectrogram(x_1khz, fs, N)
Sxx_dBFS = 10 * np.log10(Sxx + 1e-10)

# plot results
plt.figure(figsize=(14, 5))

# left subplot: single time slice showing spectral peak
plt.subplot(1, 2, 1)
mid_column = Sxx.shape[1] // 2
plt.plot(f, Sxx_dBFS[:, mid_column], 'b-', linewidth=2)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density (dBFS)')
plt.title(f'Single Time Slice of Spectrogram (t = {t[mid_column]:.3f} s)')
plt.grid(True, alpha=0.3)
plt.xlim([0, 5000])
plt.ylim([-80, 5])

peak_idx = np.argmax(Sxx_dBFS[:, mid_column])
peak_freq = f[peak_idx]
peak_amp = Sxx_dBFS[peak_idx, mid_column]
plt.axvline(peak_freq, color='r', linestyle='--', alpha=0.5, label=f'Peak: {peak_freq:.1f} Hz, {peak_amp:.1f} dBFS')
plt.legend()

# right subplot - full spectrogram
plt.subplot(1, 2, 2)
plt.pcolormesh(t, f, Sxx_dBFS, shading='gouraud', cmap='viridis', vmin=-80, vmax=0)
plt.colorbar(label='Power Spectral Density (dBFS)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram: 1 kHz Sinusoid')
plt.ylim([0, 5000])

plt.tight_layout()
plt.show()

print(f"Peak frequency: {peak_freq:.1f} Hz (expected: {f0} Hz)")
print(f"Peak amplitude: {peak_amp:.2f} dBFS (expected: ~0 dBFS for full-scale signal)")

### Test with Real Audio: Oboe Recording

In [ ]:
# read and prepare oboe audio file
fs_oboe, x_oboe_raw = wavfile.read('14.wav')

if len(x_oboe_raw.shape) > 1:
    x_oboe_raw = x_oboe_raw[:, 0]

num_samples_10s = int(10 * fs_oboe)
x_oboe_raw = x_oboe_raw[:num_samples_10s]
x_oboe = x_oboe_raw.astype(float) / (2**15)

# spectrogram
N_oboe = 4096
f_oboe, t_oboe, Sxx_oboe = spectrogram(x_oboe, fs_oboe, N_oboe)
Sxx_oboe_dBFS = 10 * np.log10(Sxx_oboe + 1e-10)

# visualizations
plt.figure(figsize=(14, 10))

# top subplot: single time slice
plt.subplot(2, 1, 1)
mid_column_oboe = Sxx_oboe.shape[1] // 2
plt.plot(f_oboe, Sxx_oboe_dBFS[:, mid_column_oboe], 'b-', linewidth=1.5)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density (dBFS)')
plt.title(f'Single Time Slice of Oboe Spectrogram (t = {t_oboe[mid_column_oboe]:.3f} s)')
plt.grid(True, alpha=0.3)
plt.xlim([0, 8000])
plt.ylim([-100, 0])
# bottom subplot: full spectrogram
plt.subplot(2, 1, 2)
plt.pcolormesh(t_oboe, f_oboe, Sxx_oboe_dBFS, shading='gouraud', cmap='viridis', vmin=-100, vmax=0)
plt.colorbar(label='Power Spectral Density (dBFS)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Spectrogram: Oboe (SQAM 14.wav, first 10s)')
plt.ylim([0, 8000])

plt.tight_layout()
plt.show()

---
## Problem 2: Peak Detection

### Mathematical Background:

For each time slice (column of Sxx), we want to identify the dominant spectral peaks. This involves:

1. **Peak finding:** Identify local maxima in the power spectral density
2. **Peak refinement:** For each peak at index $i$, combine with neighbors to get better estimates:

**Integrated peak amplitude:**
$$peakAmp[i] = psd[i-1] + psd[i] + psd[i+1]$$

**Weighted average frequency (parabolic interpolation):**
$$peakFreq[i] = \frac{psd[i-1] \cdot f[i-1] + psd[i] \cdot f[i] + psd[i+1] \cdot f[i+1]}{psd[i-1] + psd[i] + psd[i+1]}$$

This gives sub-bin frequency resolution by estimating the true peak location between DFT bins.

### Part a: Create Peak Detection Function

Create a function that:
- Takes one column of power spectral density and the frequency vector
- Finds dominant peaks using `scipy.signal.find_peaks()`
- Refines peak estimates by combining with neighbors
- Returns `(peakAmp, peakFreq)` tuples

In [ ]:
def find_spectral_peaks(psd, f, height_threshold_dB=-40, prominence=10):
   
    # convert to dB for peak finding
    psd_dB = 10 * np.log10(psd + 1e-10)

    # find peaks with minimum height and prominence requirements
    peak_indices, properties = find_peaks(psd_dB, height=height_threshold_dB, prominence=prominence)

    if len(peak_indices) == 0:
        return np.array([]), np.array([])

    # refine peak estimates by integrating with neighbors
    peakAmp = []
    peakFreq = []

    for idx in peak_indices:
        if idx == 0 or idx == len(psd) - 1:
            peakAmp.append(psd[idx])
            peakFreq.append(f[idx])
        else:
            # integrate power over peak and neighbors
            integrated_amp = psd[idx-1] + psd[idx] + psd[idx+1]
            peakAmp.append(integrated_amp)

            # weighted average frequency (parabolic interpolation)
            weighted_freq = (psd[idx-1] * f[idx-1] + psd[idx] * f[idx] + psd[idx+1] * f[idx+1]) / integrated_amp
            peakFreq.append(weighted_freq)

    return np.array(peakAmp), np.array(peakFreq)

### Part b: Apply Peak Detection to Oboe Spectrogram

Use the peak detection function on each time slice of the oboe spectrogram and visualize the results as a scatter plot showing peak trajectories over time.

In [ ]:
# peak detection to each time slice using list comprehension
peaks_over_time = [find_spectral_peaks(Sxx_oboe[:, col], f_oboe, height_threshold_dB=-50, prominence=5)
                   for col in range(Sxx_oboe.shape[1])]

# visualization comparing spectrogram with peak scatter plot
plt.figure(figsize=(14, 10))

# top subplot: original spectrogram for reference
plt.subplot(2, 1, 1)
plt.pcolormesh(t_oboe, f_oboe, Sxx_oboe_dBFS, shading='gouraud', cmap='viridis', vmin=-100, vmax=0)
plt.colorbar(label='Power Spectral Density (dBFS)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Original Spectrogram: Oboe')
plt.ylim([0, 8000])
# Bottom subplot: peak detection scatter plot
plt.subplot(2, 1, 2)

for time_idx, (peak_amps, peak_freqs) in enumerate(peaks_over_time):
    if len(peak_freqs) > 0:
        peak_amps_dB = 10 * np.log10(peak_amps + 1e-10)
        point_sizes = np.clip((peak_amps_dB + 100) / 100 * 100, 1, 100)
        plt.scatter([t_oboe[time_idx]] * len(peak_freqs), peak_freqs,
                   s=point_sizes, c='blue', alpha=0.6, edgecolors='none')

plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.title('Peak Detection Results: Dominant Spectral Peaks Over Time')
plt.ylim([0, 8000])
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# print stats
total_peaks = sum(len(peak_freqs) for _, peak_freqs in peaks_over_time)
avg_peaks_per_frame = total_peaks / len(peaks_over_time)
print(f"\nPeak Detection Statistics:")
print(f"Total peaks detected: {total_peaks}")
print(f"Average peaks per time frame: {avg_peaks_per_frame:.2f}")
print(f"Number of time frames analyzed: {len(peaks_over_time)}")